# Lab Final : เทรน YOLO แบบ Rectangle (กรอบสี่เหลี่ยม / Object Detection)
โครงเดียวกับ `lab_2box.ipynb` เดิม: **import → ตั้งค่า → เช็คเครื่อง → แปลง label → โหลดโมเดล → ลอง Predict → เทรน → ดูผล → Predict ด้วย weight ใหม่**

| | |
|---|---|
| โมเดลตั้งต้น | `yolo26n.pt` |
| ไฟล์ dataset | `rect.yaml` → `datasets/yolo_rect/` |
| คลาส | 0 airplane · 1 boat · 2 car · 3 motorcycle |
| ผลลัพธ์ | `runs/rect/` (ผลเทรน) · `weights/rect_best.pt` (weight) · `runs/predict_rect/` (ผล predict) |

> ⚠️ ต้องเปิด notebook นี้จากโฟลเดอร์ `final_dip/` เพราะ path ทุกตัวอ้างอิงจากโฟลเดอร์นี้

In [ ]:
# ===== Cell 1 : import ไลบรารี (เหมือน lab) =====
from IPython import display      # ใช้ล้างหน้าจอ output
import ultralytics               # ไลบรารี YOLO
import cv2                       # OpenCV อ่าน/วาดภาพ
import matplotlib.pyplot as plt  # แสดงภาพใน notebook
import os, glob, shutil

In [ ]:
# ===== Cell 2 : แก้ error "OMP: Error #15" บน Windows (เหมือน lab) =====
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
# ===== Cell 3 : เช็คเวอร์ชัน / CPU / GPU =====
display.clear_output()
ultralytics.checks()

## STEP 1 : เตรียม dataset (แปลง JSON ของ AnyLabeling → YOLO txt)
- ภาพและ label แบบ AnyLabeling อยู่ที่ `datasets/anylabeling_rectangle/<class>/`
- สคริปต์จะแปลงเป็นรูปแบบ YOLO และแบ่ง **train 70% / val 20% / test 10%** ไปไว้ที่ `datasets/yolo_rect/`
- ถ้าวาด label เองใน AnyLabeling ให้เปลี่ยน `--src` เป็นโฟลเดอร์ของเรา

In [ ]:
# ===== Cell 4 : แปลง label =====
!python scripts/02_anylabeling_to_yolo.py --mode rect --src datasets/anylabeling_rectangle --dst datasets/yolo_rect

In [ ]:
# ===== Cell 5 : ดูหน้าตาไฟล์ label 1 ไฟล์ =====
# Rectangle : class  x_center  y_center  width  height   (normalize 0-1)
f = sorted(glob.glob("datasets/yolo_rect/train/labels/*.txt"))[0]
print(f)
print(open(f).read()[:300])

## STEP 2 : โหลดโมเดล pretrained แล้วลอง Predict ก่อนเทรน (เหมือน lab)

In [ ]:
# ===== Cell 6 : โหลดโมเดล =====
# yolo26n.yaml = โครงสร้างโมเดล, .load('yolo26n.pt') = ใส่ weight ที่ฝึกบน COCO มาแล้ว (80 คลาส)
model = ultralytics.YOLO('yolo26n.yaml').load('yolo26n.pt')

In [ ]:
# ===== Cell 7 : ลอง Predict ด้วยโมเดลก่อนเทรน (ตอนนี้ยังรู้จัก 80 คลาสของ COCO) =====
path = sorted(glob.glob('datasets/yolo_rect/test/images/*.jpg'))[0]
im_src = cv2.imread(path)
result = model.predict(source=im_src)
im_result = cv2.cvtColor(im_src, cv2.COLOR_BGR2RGB)   # OpenCV ใช้ BGR แต่ matplotlib ใช้ RGB

for box in result[0].boxes:
    class_id = int(box.cls[0].item())                 # เลขคลาส
    conf = round(box.conf[0].item()*100, 2)           # ความมั่นใจ (%)
    display_text = f"{result[0].names[class_id]} {conf}%"
    coor = box.xyxy[0].tolist()                       # [x1, y1, x2, y2]
    coxy = [round(x) for x in coor]
    im_result = cv2.putText(im_result, display_text, (coxy[0], coxy[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2, cv2.LINE_AA)
    im_result = cv2.rectangle(im_result, (coxy[0], coxy[1]), (coxy[2], coxy[3]), (255, 255, 0), 3)

plt.figure(figsize=(10, 7))
plt.imshow(im_result)
plt.show()

## STEP 3 : เทรน (Transfer Learning)
เทรนต่อจาก weight เดิม ให้รู้จักแค่ 4 คลาสของเรา
- `epochs=20` = ดูข้อมูลครบ 20 รอบ · `imgsz=512` = ขนาดภาพ · `patience=15` = ถ้าไม่ดีขึ้น 15 รอบติดกันจะหยุดเอง
- บน CPU ใช้เวลาประมาณ 40-60 นาที (ถ้าเทรนจากสคริปต์ไปแล้ว ข้าม cell นี้ได้)

In [ ]:
# ===== Cell 8 : เทรน =====
model = ultralytics.YOLO("yolo26n.pt")
model.train(data="rect.yaml", epochs=20, imgsz=512, batch=16, patience=15,
            project="runs", name="rect", exist_ok=True)

# copy weight ที่ดีที่สุดไปไว้ที่ weights/ ให้เว็บใช้
os.makedirs("weights", exist_ok=True)
shutil.copy("runs/rect/weights/best.pt", "weights/rect_best.pt")

## STEP 4 : ดูผลการเทรน

In [ ]:
# ===== Cell 9 : กราฟ loss / mAP + confusion matrix =====
for f, size in [("results.png", (16, 8)), ("confusion_matrix_normalized.png", (8, 8))]:
    img = cv2.cvtColor(cv2.imread(f"runs/rect/{f}"), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=size); plt.imshow(img); plt.axis("off"); plt.title(f); plt.show()

In [ ]:
# ===== Cell 10 : วัดผลกับชุด test (ภาพที่ไม่ได้ใช้เทรน) =====
m = ultralytics.YOLO("weights/rect_best.pt").val(data="rect.yaml", split="test", imgsz=512,
                                                   project="runs", name="rect_test", exist_ok=True)
print("box mAP50 =", round(m.box.map50, 3), "  mAP50-95 =", round(m.box.map, 3))

## STEP 5 : Predict ด้วย weight ที่เทรนแล้ว (โค้ดวาดผลแบบเดียวกับ lab)

In [ ]:
# ===== Cell 11 : Predict ภาพ test คลาสละ 1 ภาพ =====
model = ultralytics.YOLO("weights/rect_best.pt")

for cls in ["airplane", "boat", "car", "motorcycle"]:
    path = sorted(glob.glob(f'datasets/yolo_rect/test/images/{cls}_*.jpg'))[0]
    im_src = cv2.imread(path)
    result = model.predict(source=im_src, conf=0.25, verbose=False)
    im_result = cv2.cvtColor(im_src, cv2.COLOR_BGR2RGB)

    for box in result[0].boxes:
        class_id = int(box.cls[0].item())                 # เลขคลาส
        conf = round(box.conf[0].item()*100, 2)           # ความมั่นใจ (%)
        display_text = f"{result[0].names[class_id]} {conf}%"
        coor = box.xyxy[0].tolist()                       # [x1, y1, x2, y2]
        coxy = [round(x) for x in coor]
        im_result = cv2.putText(im_result, display_text, (coxy[0], coxy[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2, cv2.LINE_AA)
        im_result = cv2.rectangle(im_result, (coxy[0], coxy[1]), (coxy[2], coxy[3]), (255, 255, 0), 3)

    plt.figure(figsize=(8, 6)); plt.imshow(im_result); plt.title(f"label: {cls}"); plt.axis("off"); plt.show()

In [ ]:
# ===== Cell 12 : Predict ภาพของเราเอง (เปลี่ยน path ได้) =====
path = 'my_test.jpg'          # <-- ใส่ภาพที่หามาเอง
if os.path.exists(path):
    result = model.predict(source=path, conf=0.25)
    plt.figure(figsize=(10, 7)); plt.imshow(result[0].plot()[:, :, ::-1]); plt.axis("off"); plt.show()
else:
    print("ไม่พบไฟล์", path)

In [ ]:
# ===== Cell 13 : เซฟผล predict ภาพ test ทั้งหมด -> runs/predict_rect/ (ใช้ส่งงาน) =====
model.predict(source="datasets/yolo_rect/test/images", conf=0.25, save=True,
              project="runs", name="predict_rect", exist_ok=True, verbose=False)
print("saved", len(glob.glob("runs/predict_rect/*.jpg")), "images -> runs/predict_rect/")